<a href="https://colab.research.google.com/github/vaibhavjiyer87/engine-nvh-deep-learning/blob/main/notebooks/11_hybrid_multitask_rpm_torque_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 1 — Mount Google Drive

# ============================================================
# CELL 1 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Persistent project location in Google Drive
PROJECT_DRIVE = Path(
    "/content/drive/MyDrive/"
    "NVH_DeepLearning/01_EngineOperatingState"
)

if not PROJECT_DRIVE.exists():
    raise FileNotFoundError(
        f"Project folder was not found:\n{PROJECT_DRIVE}"
    )

print("Google Drive mounted successfully.")
print(f"Project Drive: {PROJECT_DRIVE}")

Mounted at /content/drive
Google Drive mounted successfully.
Project Drive: /content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState


In [3]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 2 — Authenticate GitHub + determine Git author identity
# This assumes your GitHub token is stored in Colab Secrets under:
# GITHUB_TOKEN

# ============================================================
# CELL 2 — AUTHENTICATE GITHUB
# ============================================================

from google.colab import userdata

import os
import shutil
import subprocess


# ------------------------------------------------------------
# Repository settings
# ------------------------------------------------------------

REPOSITORY_NAME = "engine-nvh-deep-learning"


# ------------------------------------------------------------
# Load GitHub token securely from Colab Secrets
# ------------------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError(
        "GITHUB_TOKEN was not found in Colab Secrets.\n"
        "Add the token and enable Notebook access."
    )

# GitHub CLI recognizes GH_TOKEN automatically.
os.environ["GH_TOKEN"] = github_token
os.environ["GH_HOST"] = "github.com"


# ------------------------------------------------------------
# Install GitHub CLI if necessary
# ------------------------------------------------------------

if shutil.which("gh") is None:

    print("Installing GitHub CLI...")

    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True,
    )

    subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            "-qq",
            "gh",
        ],
        check=True,
    )


# ------------------------------------------------------------
# Verify authentication
# ------------------------------------------------------------

auth_result = subprocess.run(
    [
        "gh",
        "api",
        "user",
        "--jq",
        ".login",
    ],
    capture_output=True,
    text=True,
)

if auth_result.returncode != 0:

    print(auth_result.stderr)

    raise RuntimeError(
        "GitHub authentication failed."
    )


GITHUB_USERNAME = auth_result.stdout.strip()

# ------------------------------------------------------------
# Configure Git commit identity
# ------------------------------------------------------------

user_id_result = subprocess.run(
    [
        "gh",
        "api",
        "user",
        "--jq",
        ".id",
    ],
    capture_output=True,
    text=True,
    check=True,
)

GITHUB_USER_ID = (
    user_id_result.stdout.strip()
)

GIT_NAME = GITHUB_USERNAME

GIT_EMAIL = (
    f"{GITHUB_USER_ID}+"
    f"{GITHUB_USERNAME}@users.noreply.github.com"
)

print("Git identity prepared.")
print(f"Name:  {GIT_NAME}")
print(f"Email: {GIT_EMAIL}")

# ------------------------------------------------------------
# Configure Git to use GitHub CLI authentication
# ------------------------------------------------------------

subprocess.run(
    [
        "gh",
        "auth",
        "setup-git",
        "--hostname",
        "github.com",
        "--force",
    ],
    check=True,
)


print("GitHub authentication successful.")
print(f"GitHub user: {GITHUB_USERNAME}")
print(f"Repository:  {REPOSITORY_NAME}")

Git identity prepared.
Name:  vaibhavjiyer87
Email: 312107027+vaibhavjiyer87@users.noreply.github.com
GitHub authentication successful.
GitHub user: vaibhavjiyer87
Repository:  engine-nvh-deep-learning


In [4]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 3 — Clone repository if .git is missing / Update repository
# This cell deals with the temporary nature of /content.

# ============================================================
# CELL 3 — RESTORE LOCAL GITHUB REPOSITORY
# ============================================================

from pathlib import Path
import shutil
import subprocess


TEMP_REPO_DIR = (
    Path("/content")
    / REPOSITORY_NAME
)

REPOSITORY_IDENTIFIER = (
    f"{GITHUB_USERNAME}/"
    f"{REPOSITORY_NAME}"
)


# ------------------------------------------------------------
# Case 1:
# Valid Git repository already exists
# ------------------------------------------------------------

if (
    TEMP_REPO_DIR.exists()
    and
    (TEMP_REPO_DIR / ".git").exists()
):

    print(
        "Git repository already exists "
        "in this Colab runtime."
    )

    # Pull updates only when the working tree is clean.
    status_result = subprocess.run(
        [
            "git",
            "-C",
            str(TEMP_REPO_DIR),
            "status",
            "--porcelain",
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    if status_result.stdout.strip():

        print(
            "Local changes detected."
        )

        print(
            "Automatic git pull skipped "
            "to avoid overwriting local work."
        )

    else:

        print(
            "Working tree is clean. "
            "Updating from GitHub..."
        )

        pull_result = subprocess.run(
            [
                "git",
                "-C",
                str(TEMP_REPO_DIR),
                "pull",
                "--ff-only",
            ],
            capture_output=True,
            text=True,
        )

        print(pull_result.stdout)

        if pull_result.returncode != 0:
            print(pull_result.stderr)


# ------------------------------------------------------------
# Case 2:
# Folder exists, but it is NOT a Git repository
# ------------------------------------------------------------

elif TEMP_REPO_DIR.exists():

    raise RuntimeError(
        f"The folder exists but is not a Git repository:\n"
        f"{TEMP_REPO_DIR}\n\n"
        "Do not run git init. Inspect or back up the folder "
        "before removing it and rerunning this cell."
    )


# ------------------------------------------------------------
# Case 3:
# Fresh runtime — clone repository
# ------------------------------------------------------------

else:

    print(
        "Repository not present in this runtime."
    )

    print(
        f"Cloning {REPOSITORY_IDENTIFIER}..."
    )

    clone_result = subprocess.run(
        [
            "gh",
            "repo",
            "clone",
            REPOSITORY_IDENTIFIER,
            str(TEMP_REPO_DIR),
        ],
        capture_output=True,
        text=True,
    )

    print(clone_result.stdout)

    if clone_result.returncode != 0:

        print(clone_result.stderr)

        raise RuntimeError(
            "Repository clone failed."
        )


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

if not (
    TEMP_REPO_DIR
    / ".git"
).exists():

    raise RuntimeError(
        "Repository restoration failed."
    )


print("Local Git repository is ready.")
print(f"Location: {TEMP_REPO_DIR}")

Repository not present in this runtime.
Cloning vaibhavjiyer87/engine-nvh-deep-learning...

Local Git repository is ready.
Location: /content/engine-nvh-deep-learning


In [5]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 4 — Define REPO_DIR and all standard paths + restore requirements

# ============================================================
# CELL 4 — DEFINE PROJECT PATHS
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# GitHub working repository
# ------------------------------------------------------------

REPO_DIR = (
    Path("/content")
    / REPOSITORY_NAME
)

if not (
    REPO_DIR
    / ".git"
).exists():

    raise FileNotFoundError(
        f"Valid Git repository not found at:\n"
        f"{REPO_DIR}\n\n"
        "Run Cell 3 first."
    )


# ------------------------------------------------------------
# Persistent raw dataset
# ------------------------------------------------------------

RAW_ROOT = (
    PROJECT_DRIVE
    / "data"
    / "raw"
    / "procedural_engine_sounds"
)

DATASET_ROOT = (
    RAW_ROOT
    / "dataset"
)

AUDIO_DIR = (
    DATASET_ROOT
    / "audio"
    / "A_full_set"
)


# ------------------------------------------------------------
# Persistent Google Drive manifests
# ------------------------------------------------------------

DRIVE_MANIFEST_DIR = (
    PROJECT_DRIVE
    / "data"
    / "manifests"
)


# ------------------------------------------------------------
# GitHub configuration
# ------------------------------------------------------------

CONFIG_DIR = (
    REPO_DIR
    / "configs"
)

REPORT_DIR = (
    REPO_DIR
    / "reports"
)

SPLIT_DIR = (
    REPO_DIR
    / "data"
    / "splits"
)


# ------------------------------------------------------------
# GitHub result directories
# ------------------------------------------------------------

RESULTS_DIR = (
    REPO_DIR
    / "results"
)

FIGURE_DIR = (
    RESULTS_DIR
    / "figures"
)

TABLE_DIR = (
    RESULTS_DIR
    / "tables"
)

DATA_AUDIT_FIGURE_DIR = (
    FIGURE_DIR
    / "data_audit"
)

DESIGN_FIGURE_DIR = (
    FIGURE_DIR
    / "preprocessing_design"
)

SPLIT_FIGURE_DIR = (
    FIGURE_DIR
    / "split_design"
)


# ------------------------------------------------------------
# Persistent Drive output directories
# ------------------------------------------------------------

DRIVE_OUTPUT_DIR = (
    PROJECT_DRIVE
    / "outputs"
)

DRIVE_DESIGN_TABLE_DIR = (
    DRIVE_OUTPUT_DIR
    / "tables"
    / "preprocessing_design"
)


# ------------------------------------------------------------
# Create output folders if missing
# ------------------------------------------------------------

directories_to_create = [
    CONFIG_DIR,
    REPORT_DIR,
    SPLIT_DIR,
    FIGURE_DIR,
    TABLE_DIR,
    DATA_AUDIT_FIGURE_DIR,
    DESIGN_FIGURE_DIR,
    SPLIT_FIGURE_DIR,
    DRIVE_DESIGN_TABLE_DIR,
]

for directory in directories_to_create:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print("Project paths restored.")
print()
print(f"REPO_DIR:     {REPO_DIR}")
print(f"PROJECT_DRIVE:{PROJECT_DRIVE}")
print(f"RAW_ROOT:     {RAW_ROOT}")
print(f"AUDIO_DIR:    {AUDIO_DIR}")

# ------------------------------------------------------------
# Apply Git commit identity to cloned repository
# ------------------------------------------------------------

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "config",
        "user.name",
        GIT_NAME,
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "config",
        "user.email",
        GIT_EMAIL,
    ],
    check=True,
)

print("Git commit identity applied to repository.")

# ------------------------------------------------------------
# Restore project Python dependencies
# ------------------------------------------------------------

import subprocess

REQUIREMENTS_PATH = (
    REPO_DIR
    / "requirements.txt"
)

if not REQUIREMENTS_PATH.exists():
    raise FileNotFoundError(
        f"requirements.txt not found:\n"
        f"{REQUIREMENTS_PATH}"
    )

install_result = subprocess.run(
    [
        "python",
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REQUIREMENTS_PATH),
    ],
    capture_output=True,
    text=True,
)

if install_result.returncode != 0:

    print(install_result.stdout)
    print(install_result.stderr)

    raise RuntimeError(
        "Project dependency installation failed."
    )

print(
    "Project Python dependencies restored."
)

Project paths restored.

REPO_DIR:     /content/engine-nvh-deep-learning
PROJECT_DRIVE:/content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState
RAW_ROOT:     /content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState/data/raw/procedural_engine_sounds
AUDIO_DIR:    /content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState/data/raw/procedural_engine_sounds/dataset/audio/A_full_set
Git commit identity applied to repository.
Project Python dependencies restored.


In [6]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 5 — Load persistent analysis data
# This cell restores the major tables you've created so far.
# This is conditional, as some files may not exist yet depending on
# where you are in the project.

# ============================================================
# CELL 5 — LOAD PERSISTENT ANALYSIS DATA
# ============================================================

# ============================================================
# STANDARD PROJECT IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import yaml

from tqdm.auto import tqdm

print(
    "Standard project libraries imported."
)


# ------------------------------------------------------------
# 1. Raw-file manifest — REQUIRED
# ------------------------------------------------------------

RAW_MANIFEST_PATH = (
    DRIVE_MANIFEST_DIR
    / "raw_file_manifest_v001.csv"
)

if not RAW_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        f"Required raw manifest not found:\n"
        f"{RAW_MANIFEST_PATH}"
    )


raw_manifest = pd.read_csv(
    RAW_MANIFEST_PATH
)

print(
    f"LOADED raw_manifest: "
    f"{raw_manifest.shape}"
)


# ------------------------------------------------------------
# 2. PREP-001 detailed target analysis — OPTIONAL
# ------------------------------------------------------------

PREP001_TARGET_PATH = (
    DRIVE_DESIGN_TABLE_DIR
    / "prep001_target_analysis_v001.csv.gz"
)

if PREP001_TARGET_PATH.exists():

    prep001_targets = pd.read_csv(
        PREP001_TARGET_PATH
    )

    print(
        f"LOADED prep001_targets: "
        f"{prep001_targets.shape}"
    )

else:

    prep001_targets = None

    print(
        "NOT FOUND: prep001 target analysis "
        "(this is okay if it has not been generated yet)."
    )


# ------------------------------------------------------------
# 3. Window candidate metrics — OPTIONAL
# ------------------------------------------------------------

WINDOW_CANDIDATE_PATH = (
    DRIVE_DESIGN_TABLE_DIR
    / "window_candidate_metrics_v001.csv.gz"
)

if WINDOW_CANDIDATE_PATH.exists():

    window_candidates = pd.read_csv(
        WINDOW_CANDIDATE_PATH
    )

    print(
        f"LOADED window_candidates: "
        f"{window_candidates.shape}"
    )

else:

    window_candidates = None

    print(
        "NOT FOUND: window candidate metrics."
    )


# ------------------------------------------------------------
# 4. Frozen SPLIT-001 file assignment — OPTIONAL
# ------------------------------------------------------------

FILE_SPLIT_PATH = (
    SPLIT_DIR
    / "file_split_v001.csv"
)

if FILE_SPLIT_PATH.exists():

    file_split = pd.read_csv(
        FILE_SPLIT_PATH
    )

    print(
        f"LOADED file_split: "
        f"{file_split.shape}"
    )

else:

    file_split = None

    print(
        "NOT FOUND: SPLIT-001 file assignment."
    )


# ============================================================
# LOAD REQUIRED FROZEN PROJECT SPECIFICATIONS
# ============================================================

import yaml


# ------------------------------------------------------------
# PREP-001 — REQUIRED
# ------------------------------------------------------------

PREPROCESSING_CONFIG_PATH = (
    CONFIG_DIR
    / "preprocessing_v001.yaml"
)

if not PREPROCESSING_CONFIG_PATH.exists():

    raise FileNotFoundError(
        "Required PREP-001 configuration is missing:\n"
        f"{PREPROCESSING_CONFIG_PATH}"
    )


try:

    with open(
        PREPROCESSING_CONFIG_PATH,
        "r",
        encoding="utf-8",
    ) as file:

        preprocessing_config = (
            yaml.safe_load(file)
        )

except yaml.YAMLError as error:

    raise RuntimeError(
        "PREP-001 exists but is invalid YAML.\n"
        f"{error}"
    )


if (
    preprocessing_config[
        "specification"
    ]["preprocessing_version"]
    != "PREP-001"
):

    raise RuntimeError(
        "Unexpected preprocessing version."
    )


print(
    "LOADED preprocessing_config: PREP-001"
)


# ------------------------------------------------------------
# SPLIT-001 — REQUIRED
# ------------------------------------------------------------

SPLIT_CONFIG_PATH = (
    CONFIG_DIR
    / "split_v001.yaml"
)

if not SPLIT_CONFIG_PATH.exists():

    raise FileNotFoundError(
        "Required SPLIT-001 configuration is missing:\n"
        f"{SPLIT_CONFIG_PATH}"
    )


try:

    with open(
        SPLIT_CONFIG_PATH,
        "r",
        encoding="utf-8",
    ) as file:

        split_config = (
            yaml.safe_load(file)
        )

except yaml.YAMLError as error:

    raise RuntimeError(
        "SPLIT-001 exists but is invalid YAML.\n"
        f"{error}"
    )


if (
    split_config[
        "specification"
    ]["split_version"]
    != "SPLIT-001"
):

    raise RuntimeError(
        "Unexpected split version."
    )


print(
    "LOADED split_config: SPLIT-001"
)


# ------------------------------------------------------------
# FILE SPLIT — REQUIRED
# ------------------------------------------------------------

FILE_SPLIT_PATH = (
    SPLIT_DIR
    / "file_split_v001.csv"
)

if not FILE_SPLIT_PATH.exists():

    raise FileNotFoundError(
        "Required SPLIT-001 assignment file is missing:\n"
        f"{FILE_SPLIT_PATH}"
    )


file_split = pd.read_csv(
    FILE_SPLIT_PATH
)

print(
    f"LOADED file_split: "
    f"{file_split.shape}"
)


# ============================================================
# DERIVE FROZEN RUNTIME CONSTANTS
# ============================================================

# ------------------------------------------------------------
# PREP-001 identity
# ------------------------------------------------------------

PREPROCESSING_VERSION = (
    preprocessing_config[
        "specification"
    ]["preprocessing_version"]
)

DATASET_SUBSET = (
    preprocessing_config[
        "specification"
    ]["dataset_subset"]
)


# ------------------------------------------------------------
# Windowing
# ------------------------------------------------------------

WINDOW_DURATION_S = float(
    preprocessing_config[
        "windowing"
    ]["window_duration_s"]
)

OVERLAP_FRACTION = float(
    preprocessing_config[
        "windowing"
    ]["overlap_fraction"]
)

HOP_DURATION_S = float(
    preprocessing_config[
        "windowing"
    ]["hop_duration_s"]
)


# ------------------------------------------------------------
# Source-data definition
# ------------------------------------------------------------

SOURCE_SAMPLE_RATE_HZ = int(
    preprocessing_config[
        "source_data"
    ]["source_sample_rate_hz"]
)

EXPECTED_CHANNELS = int(
    preprocessing_config[
        "source_data"
    ]["expected_channels"]
)

RPM_SCALE_FACTOR = float(
    preprocessing_config[
        "source_data"
    ]["rpm_scale_factor"]
)

TORQUE_SCALE_FACTOR_NM = float(
    preprocessing_config[
        "source_data"
    ]["torque_scale_factor_nm"]
)


# ------------------------------------------------------------
# YAML channel numbers are human-readable 1-based numbers.
# NumPy arrays use 0-based indexing.
# ------------------------------------------------------------

RPM_CHANNEL_INDEX = (
    int(
        preprocessing_config[
            "source_data"
        ]["annotation_channels"]["rpm"]
    )
    - 1
)

TORQUE_CHANNEL_INDEX = (
    int(
        preprocessing_config[
            "source_data"
        ]["annotation_channels"]["torque"]
    )
    - 1
)


# ------------------------------------------------------------
# Audio processing
# ------------------------------------------------------------

TARGET_SAMPLE_RATE_HZ = int(
    preprocessing_config[
        "audio_processing"
    ]["target_sample_rate_hz"]
)

AUDIO_CHANNEL_STRATEGY = (
    preprocessing_config[
        "audio_processing"
    ]["channel_strategy"]
)


# ------------------------------------------------------------
# Steady-state criterion
# ------------------------------------------------------------

ABSOLUTE_RPM_LIMIT = float(
    preprocessing_config[
        "operating_state"
    ]["steady_state_rule"][
        "maximum_absolute_range_rpm"
    ]
)

RELATIVE_RPM_LIMIT = float(
    preprocessing_config[
        "operating_state"
    ]["steady_state_rule"][
        "maximum_relative_range_fraction"
    ]
)


# ------------------------------------------------------------
# SPLIT-001 identity
# ------------------------------------------------------------

SPLIT_VERSION = (
    split_config[
        "specification"
    ]["split_version"]
)


print("Frozen runtime constants restored.")
print()
print(f"Preprocessing:       {PREPROCESSING_VERSION}")
print(f"Split:               {SPLIT_VERSION}")
print(f"Dataset subset:      {DATASET_SUBSET}")
print(f"Window duration:     {WINDOW_DURATION_S} s")
print(f"Overlap:             {100 * OVERLAP_FRACTION:.0f}%")
print(f"Source sample rate:  {SOURCE_SAMPLE_RATE_HZ} Hz")
print(f"Target sample rate:  {TARGET_SAMPLE_RATE_HZ} Hz")
print(f"Expected channels:   {EXPECTED_CHANNELS}")
print(f"RPM channel index:   {RPM_CHANNEL_INDEX}")
print(f"Torque channel idx:  {TORQUE_CHANNEL_INDEX}")
print(f"Channel strategy:    {AUDIO_CHANNEL_STRATEGY}")

# ============================================================
# VALIDATE FROZEN RUNTIME CONSTANTS
# ============================================================

assert PREPROCESSING_VERSION == "PREP-001", (
    f"Expected PREP-001, found {PREPROCESSING_VERSION}"
)

assert SPLIT_VERSION == "SPLIT-001", (
    f"Expected SPLIT-001, found {SPLIT_VERSION}"
)

assert SOURCE_SAMPLE_RATE_HZ == 48000, (
    "Unexpected PREP-001 source sample rate."
)

assert TARGET_SAMPLE_RATE_HZ == 16000, (
    "Unexpected PREP-001 target sample rate."
)

assert EXPECTED_CHANNELS == 4, (
    "Unexpected PREP-001 channel count."
)

assert WINDOW_DURATION_S > 0

assert 0 <= OVERLAP_FRACTION < 1

assert RPM_CHANNEL_INDEX < EXPECTED_CHANNELS

assert TORQUE_CHANNEL_INDEX < EXPECTED_CHANNELS

assert file_split["file_id"].is_unique

assert set(
    file_split["split"].unique()
) == {
    "train",
    "validation",
    "test",
}

print(
    "PASS: Frozen runtime constants validated."
)

# ============================================================
# LOAD SAMPLE-MANIFEST-001
# ============================================================

SAMPLE_MANIFEST_PATH = (
    DRIVE_MANIFEST_DIR
    / "sample_manifest_v001.csv"
)

if not SAMPLE_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        "Required SAMPLE-MANIFEST-001 "
        "is missing:\n"
        f"{SAMPLE_MANIFEST_PATH}"
    )


sample_manifest = pd.read_csv(
    SAMPLE_MANIFEST_PATH
)


assert (
    sample_manifest[
        "sample_id"
    ].is_unique
)

assert set(
    sample_manifest[
        "preprocessing_version"
    ]
) == {
    "PREP-001"
}

assert set(
    sample_manifest[
        "split_version"
    ]
) == {
    "SPLIT-001"
}


print(
    f"LOADED sample_manifest: "
    f"{sample_manifest.shape}"
)

# ============================================================
# PROJECT SESSION READINESS SUMMARY
# ============================================================

print()
print("=" * 64)
print("NVH DEEP-LEARNING PROJECT SESSION READY")
print("=" * 64)

print(
    f"Raw manifest:       "
    f"{len(raw_manifest):,} source files"
)

print(
    f"Preprocessing:      "
    f"{PREPROCESSING_VERSION}"
)

print(
    f"Dataset split:      "
    f"{SPLIT_VERSION}"
)

print(
    f"Window:             "
    f"{WINDOW_DURATION_S:.1f} s"
)

print(
    f"Overlap:            "
    f"{100 * OVERLAP_FRACTION:.0f}%"
)

print(
    f"Source sample rate: "
    f"{SOURCE_SAMPLE_RATE_HZ:,} Hz"
)

print(
    f"Model sample rate:  "
    f"{TARGET_SAMPLE_RATE_HZ:,} Hz"
)

print(
    f"Audio strategy:     "
    f"{AUDIO_CHANNEL_STRATEGY}"
)

print()
print("SPLIT-001:")

print(
    file_split[
        "split"
    ]
    .value_counts()
    .to_string()
)

if (
    "sample_manifest" in globals()
    and
    sample_manifest is not None
):

    print()
    print(
        f"Sample manifest:    "
        f"{len(sample_manifest):,} samples"
    )

else:

    print()
    print(
        "Sample manifest:    "
        "not generated yet"
    )

print("=" * 64)

print(
    f"Sample manifest:    "
    f"{len(sample_manifest):,} samples"
)

# ============================================================
# LOAD ORDER-001 (added after initiating 06_bseline_feature_generation.ipynb)
# ============================================================

ORDER_CONFIG_PATH = (
    CONFIG_DIR
    / "order_analysis_v001.yaml"
)

if not ORDER_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"ORDER-001 configuration not found:\n"
        f"{ORDER_CONFIG_PATH}"
    )

with open(
    ORDER_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    order_analysis_config = yaml.safe_load(file)

ORDER_ANALYSIS_VERSION = (
    order_analysis_config[
        "specification"
    ]["order_analysis_version"]
)

assert ORDER_ANALYSIS_VERSION == "ORDER-001"

print(
    "PASS: ORDER-001 loaded."
)

# ============================================================
# LOAD FEAT-001 — REQUIRED FROM NOTEBOOK 07 ONWARD
# ============================================================

FEATURE_CONFIG_PATH = (
    CONFIG_DIR
    / "features_v001.yaml"
)

if not FEATURE_CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Required FEAT-001 configuration "
        "is missing:\n"
        f"{FEATURE_CONFIG_PATH}"
    )


try:

    with open(
        FEATURE_CONFIG_PATH,
        "r",
        encoding="utf-8",
    ) as file:

        feature_config = yaml.safe_load(
            file
        )

except yaml.YAMLError as error:

    raise RuntimeError(
        "FEAT-001 exists but is invalid YAML.\n"
        f"{error}"
    )


FEATURE_VERSION = (
    feature_config[
        "specification"
    ]["feature_version"]
)

assert FEATURE_VERSION == "FEAT-001", (
    f"Expected FEAT-001, "
    f"found {FEATURE_VERSION}"
)


# ------------------------------------------------------------
# Load persistent FEAT-001 feature table
# ------------------------------------------------------------

BASELINE_FEATURE_PATH = (
    PROJECT_DRIVE
    / "data"
    / "cached_features"
    / "FEAT-001"
    / "baseline_features_v001.parquet"
)

if not BASELINE_FEATURE_PATH.exists():

    raise FileNotFoundError(
        "Required FEAT-001 feature table "
        "is missing:\n"
        f"{BASELINE_FEATURE_PATH}"
    )


baseline_features = pd.read_parquet(
    BASELINE_FEATURE_PATH
)


# ------------------------------------------------------------
# Validate frozen feature table
# ------------------------------------------------------------

assert (
    baseline_features[
        "sample_id"
    ].is_unique
)

assert (
    baseline_features[
        "sample_id"
    ].notna().all()
)

assert set(
    baseline_features[
        "feature_version"
    ].unique()
) == {
    "FEAT-001"
}

assert set(
    baseline_features[
        "split"
    ].unique()
) == {
    "train",
    "validation",
    "test",
}

assert (
    len(baseline_features)
    ==
    len(sample_manifest)
)

assert set(
    baseline_features[
        "sample_id"
    ]
) == set(
    sample_manifest[
        "sample_id"
    ]
)


print(
    f"LOADED baseline_features: "
    f"{baseline_features.shape}"
)

print(
    f"Feature version: "
    f"{FEATURE_VERSION}"
)

print(
    "PASS: FEAT-001 restored and validated."
)

# ------------------------------------------------------------
# Updating Readiness Summary
# ------------------------------------------------------------
print()
print("=" * 68)
print("MODELING ENVIRONMENT READY")
print("=" * 68)

print(
    f"Preprocessing:       "
    f"{PREPROCESSING_VERSION}"
)

print(
    f"Dataset split:       "
    f"{SPLIT_VERSION}"
)

print(
    f"Order analysis:      "
    f"{ORDER_ANALYSIS_VERSION}"
)

print(
    f"Feature definition:  "
    f"{FEATURE_VERSION}"
)

print(
    f"Sample manifest:     "
    f"{len(sample_manifest):,} samples"
)

print(
    f"Feature table:       "
    f"{len(baseline_features):,} samples"
)

print(
    f"Feature columns:     "
    f"{baseline_features.shape[1]}"
)

print("=" * 68)

Standard project libraries imported.
LOADED raw_manifest: (767, 21)
LOADED prep001_targets: (16503, 23)
LOADED window_candidates: (58653, 16)
LOADED file_split: (767, 11)
LOADED preprocessing_config: PREP-001
LOADED split_config: SPLIT-001
LOADED file_split: (767, 11)
Frozen runtime constants restored.

Preprocessing:       PREP-001
Split:               SPLIT-001
Dataset subset:      A_full_set
Window duration:     1.0 s
Overlap:             50%
Source sample rate:  48000 Hz
Target sample rate:  16000 Hz
Expected channels:   4
RPM channel index:   2
Torque channel idx:  3
Channel strategy:    mono_average
PASS: Frozen runtime constants validated.
LOADED sample_manifest: (16503, 34)

NVH DEEP-LEARNING PROJECT SESSION READY
Raw manifest:       767 source files
Preprocessing:      PREP-001
Dataset split:      SPLIT-001
Window:             1.0 s
Overlap:            50%
Source sample rate: 48,000 Hz
Model sample rate:  16,000 Hz
Audio strategy:     mono_average

SPLIT-001:
split
train      

In [7]:
# Step 93 — Create Notebook 11
# Create: notebooks/11_hybrid_multitask_rpm_torque_model.ipynb
# Save it to GitHub and run Startup Cells 1 thru 5

# Step 94 — Restore the shared MTL population and benchmarks
# Step 94.1 — Restore MTL population, normalization and benchmark evidence

# ============================================================
# STEP 94.1 — RESTORE SHARED MTL MODELING STATE
# ============================================================

from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import h5py
import joblib
import torch
import sys
import importlib

MTL_POPULATION_PATH = (
    PROJECT_DRIVE
    / "data"
    / "manifests"
    / "torque_population_v001.csv"
)

MTL001_TARGET_NORMALIZATION_PATH = (
    REPO_DIR
    / "data"
    / "mtl001_target_normalization_v001.yaml"
)

MTL_BENCHMARK_PATH = (
    REPO_DIR
    / "results"
    / "tables"
    / "multitask"
    / "mtl001_validation_benchmarks_v001.csv"
)

BASELINE_FEATURE_PATH = (
    PROJECT_DRIVE
    / "data"
    / "cached_features"
    / "FEAT-001"
    / "baseline_features_v001.parquet"
)

BASE002_MODEL_PATH = (
    PROJECT_DRIVE
    / "models"
    / "baseline"
    / "base002_random_forest_rpm_v001.joblib"
)

TBASE002_MODEL_PATH = (
    PROJECT_DRIVE
    / "models"
    / "torque_baseline"
    / "tbase002_random_forest_torque_v001.joblib"
)


for path in [
    MTL_POPULATION_PATH,
    MTL001_TARGET_NORMALIZATION_PATH,
    MTL_BENCHMARK_PATH,
    BASELINE_FEATURE_PATH,
    BASE002_MODEL_PATH,
    TBASE002_MODEL_PATH,
]:

    if not path.exists():
        raise FileNotFoundError(
            f"Required artifact missing:\n{path}"
        )


mtl_population = pd.read_csv(
    MTL_POPULATION_PATH
)

baseline_features = pd.read_parquet(
    BASELINE_FEATURE_PATH
)

mtl_benchmarks = pd.read_csv(
    MTL_BENCHMARK_PATH
)


with open(
    MTL001_TARGET_NORMALIZATION_PATH,
    "r",
    encoding="utf-8",
) as file:

    target_norm = yaml.safe_load(
        file
    )


MTL_RPM_MEAN = float(
    target_norm["rpm"]["mean"]
)

MTL_RPM_STD = float(
    target_norm["rpm"]["std"]
)

MTL_TORQUE_MEAN = float(
    target_norm["torque_nm"]["mean"]
)

MTL_TORQUE_STD = float(
    target_norm["torque_nm"]["std"]
)


base002_model = joblib.load(
    BASE002_MODEL_PATH
)

tbase002_model = joblib.load(
    TBASE002_MODEL_PATH
)


RPM_FEATURE_COLUMNS = list(
    base002_model.feature_names_in_
)

TORQUE_FEATURE_COLUMNS = list(
    tbase002_model.feature_names_in_
)


assert (
    RPM_FEATURE_COLUMNS
    ==
    TORQUE_FEATURE_COLUMNS
)

MTL002_FEATURE_COLUMNS = (
    RPM_FEATURE_COLUMNS
)

assert (
    len(MTL002_FEATURE_COLUMNS)
    == 41
)


assert (
    len(
        mtl_population.loc[
            mtl_population["split"] == "train"
        ]
    )
    == 7739
)

assert (
    len(
        mtl_population.loc[
            mtl_population["split"] == "validation"
        ]
    )
    == 1749
)

assert (
    len(
        mtl_population.loc[
            mtl_population["split"] == "test"
        ]
    )
    == 1815
)


display(
    mtl_benchmarks
)

print(
    "PASS: Shared population, targets and "
    "conventional benchmarks restored."
)

,target,benchmark_id,mae,rmse,r2,p95_absolute_error
0,rpm_mean,BASE-002,38.386810,129.145184,0.993772,188.836825
1,torque_mean_nm,TBASE-002,23.321647,41.574599,0.950266,104.451214


PASS: Shared population, targets and conventional benchmarks restored.


In [8]:
# Step 95 — Create training-only normalization for the 41 engineered features
# MTL-002's handcrafted-feature branch needs standardized inputs.
# We must calculate those statistics from the 7,739 training windows only.

# Step 95.1 — Construct the shared feature population

# ============================================================
# STEP 95.1 — BUILD MTL-002 FEATURE POPULATION
# ============================================================

mtl002_feature_population = (

    mtl_population[
        [
            "sample_id",
            "parent_file_id",
            "split",
            "rpm_mean",
            "torque_mean_nm",
        ]
    ]

    .merge(

        baseline_features[
            [
                "sample_id",
                *MTL002_FEATURE_COLUMNS,
            ]
        ],

        on="sample_id",

        how="inner",

        validate="one_to_one",
    )
)


assert (
    len(mtl002_feature_population)
    ==
    11303
)


assert not (
    mtl002_feature_population[
        MTL002_FEATURE_COLUMNS
    ]
    .isna()
    .any()
    .any()
)


print(
    f"Rows: "
    f"{len(mtl002_feature_population):,}"
)

print(
    f"Engineered predictors: "
    f"{len(MTL002_FEATURE_COLUMNS)}"
)

print(
    "PASS: Hybrid feature population assembled."
)

Rows: 11,303
Engineered predictors: 41
PASS: Hybrid feature population assembled.


In [9]:
# Step 95.2 — Calculate train-only feature statistics

# ============================================================
# STEP 95.2 — TRAINING-ONLY ENGINEERED FEATURE NORMALIZATION
# ============================================================

training_feature_population = (
    mtl002_feature_population.loc[
        mtl002_feature_population[
            "split"
        ] == "train"
    ]
)


feature_mean = (
    training_feature_population[
        MTL002_FEATURE_COLUMNS
    ]
    .mean()
)


feature_std = (
    training_feature_population[
        MTL002_FEATURE_COLUMNS
    ]
    .std(
        ddof=0
    )
)


zero_std_features = (
    feature_std[
        feature_std == 0
    ]
    .index
    .tolist()
)


if zero_std_features:

    print(
        "Zero-variance features:"
    )

    print(
        zero_std_features
    )


# Safe denominator for any zero-variance predictor.
feature_std_safe = (
    feature_std.copy()
)

feature_std_safe.loc[
    feature_std_safe == 0
] = 1.0


assert np.isfinite(
    feature_mean.to_numpy()
).all()

assert np.isfinite(
    feature_std_safe.to_numpy()
).all()


print(
    "PASS: FEAT-001 training-only "
    "normalization calculated."
)

PASS: FEAT-001 training-only normalization calculated.


In [10]:
# Step 95.3 — Persist the normalization record

# ============================================================
# STEP 95.3 — SAVE MTL-002 FEATURE NORMALIZATION
# ============================================================

MTL002_FEATURE_NORMALIZATION_PATH = (
    REPO_DIR
    / "data"
    / "mtl002_feature_normalization_v001.yaml"
)


feature_normalization_record = {

    "model_version":
        "MTL-002",

    "fit_population":
        "shared_training_split_only",

    "training_sample_count":
        7739,

    "feature_source":
        "FEAT-001",

    "feature_count":
        len(
            MTL002_FEATURE_COLUMNS
        ),

    "feature_order":
        list(
            MTL002_FEATURE_COLUMNS
        ),

    "mean": {

        feature:
            float(
                feature_mean[
                    feature
                ]
            )

        for feature in
        MTL002_FEATURE_COLUMNS
    },

    "std": {

        feature:
            float(
                feature_std_safe[
                    feature
                ]
            )

        for feature in
        MTL002_FEATURE_COLUMNS
    },

    "zero_variance_features":
        zero_std_features,

    "validation_statistics_used":
        False,

    "test_statistics_used":
        False,
}


with open(
    MTL002_FEATURE_NORMALIZATION_PATH,
    "w",
    encoding="utf-8",
) as file:

    yaml.safe_dump(
        feature_normalization_record,
        file,
        sort_keys=False,
    )


print(
    f"Saved:\n"
    f"{MTL002_FEATURE_NORMALIZATION_PATH}"
)

Saved:
/content/engine-nvh-deep-learning/data/mtl002_feature_normalization_v001.yaml


Step 96 — Create the hybrid Dataset module

The Dataset will return:

* log-mel tensor -->
* 41 normalized FEAT-001 predictors -->
* standardized RPM target -->
* standardized torque target -->
* cache index

In [11]:
# Step 96.1 — Create src/hybrid_multitask_dataset.py

# ============================================================
# STEP 96.1 — CREATE HYBRID MULTITASK DATASET MODULE
# ============================================================

import textwrap

HYBRID_DATASET_MODULE_PATH = (
    REPO_DIR
    / "src"
    / "hybrid_multitask_dataset.py"
)


hybrid_dataset_code = r'''
import h5py
import numpy as np
import torch

from torch.utils.data import Dataset


class HybridLogMelFeatureDataset(Dataset):

    def __init__(
        self,
        h5_path,
        indices,
        engineered_features,
        logmel_mean,
        logmel_std,
        rpm_mean,
        rpm_std,
        torque_mean,
        torque_std,
    ):

        self.h5_path = str(
            h5_path
        )

        self.indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        self.engineered_features = np.asarray(
            engineered_features,
            dtype=np.float32,
        )

        if (
            len(self.engineered_features)
            !=
            len(self.indices)
        ):

            raise ValueError(
                "engineered_features and indices "
                "must contain the same number of rows."
            )

        self.logmel_mean = float(
            logmel_mean
        )

        self.logmel_std = float(
            logmel_std
        )

        self.rpm_mean = float(
            rpm_mean
        )

        self.rpm_std = float(
            rpm_std
        )

        self.torque_mean = float(
            torque_mean
        )

        self.torque_std = float(
            torque_std
        )

        self._h5 = None


    def _get_h5(self):

        if self._h5 is None:

            self._h5 = h5py.File(
                self.h5_path,
                "r",
            )

        return self._h5


    def __len__(self):

        return len(
            self.indices
        )


    def __getitem__(
        self,
        dataset_index,
    ):

        h5 = self._get_h5()

        cache_index = int(
            self.indices[
                dataset_index
            ]
        )


        logmel = (
            h5[
                "log_mel"
            ][
                cache_index
            ]
            .astype(
                np.float32
            )
        )


        rpm = float(
            h5[
                "rpm_mean"
            ][
                cache_index
            ]
        )


        torque = float(
            h5[
                "torque_mean_nm"
            ][
                cache_index
            ]
        )


        logmel = (
            logmel
            - self.logmel_mean
        ) / self.logmel_std


        rpm_standardized = (
            rpm
            - self.rpm_mean
        ) / self.rpm_std


        torque_standardized = (
            torque
            - self.torque_mean
        ) / self.torque_std


        logmel_tensor = (
            torch.from_numpy(
                logmel
            )
            .unsqueeze(0)
        )


        feature_tensor = (
            torch.from_numpy(
                self.engineered_features[
                    dataset_index
                ]
            )
        )


        rpm_tensor = torch.tensor(
            rpm_standardized,
            dtype=torch.float32,
        )


        torque_tensor = torch.tensor(
            torque_standardized,
            dtype=torch.float32,
        )


        return (
            logmel_tensor,
            feature_tensor,
            rpm_tensor,
            torque_tensor,
            cache_index,
        )


    def close(self):

        if self._h5 is not None:

            self._h5.close()

            self._h5 = None


    def __del__(self):

        self.close()
'''


HYBRID_DATASET_MODULE_PATH.write_text(
    textwrap.dedent(
        hybrid_dataset_code
    ).lstrip(),
    encoding="utf-8",
)


print(
    f"Created:\n"
    f"{HYBRID_DATASET_MODULE_PATH}"
)

Created:
/content/engine-nvh-deep-learning/src/hybrid_multitask_dataset.py


Step 97 — Create the MTL-002 hybrid model

The acoustic branch remains the frequency-aware CNN used in MTL-001.

The new branch is:

* 41 engineered features         ↓
* Linear 41 → 64        ↓
* ReLU        ↓
* Linear 64 → 32         ↓
* feature embedding

Then:

* CNN embedding (64) + FEAT embedding (32)        ↓
* 96-dimensional fusion        ↓
* 64-dimensional shared representation        ↓
* RPM head ---- Torque head

In [12]:
# Step 97.1 — Create src/hybrid_multitask_models.py

# ============================================================
# STEP 97.1 — CREATE MTL-002 HYBRID MODEL
# ============================================================

HYBRID_MODEL_MODULE_PATH = (
    REPO_DIR
    / "src"
    / "hybrid_multitask_models.py"
)


hybrid_model_code = r'''
import torch
import torch.nn as nn


class HybridRPMTorqueMTL(nn.Module):

    def __init__(
        self,
        engineered_feature_count=41,
        dropout=0.20,
    ):

        super().__init__()


        # ----------------------------------------------------
        # Acoustic branch
        # Same frequency-aware encoder family used in MTL-001.
        # ----------------------------------------------------

        self.acoustic_encoder = nn.Sequential(

            nn.Conv2d(
                1, 16,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=(1, 2)
            ),


            nn.Conv2d(
                16, 32,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=(1, 2)
            ),


            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=(2, 2)
            ),

            nn.AdaptiveAvgPool2d(
                (32, 1)
            ),
        )


        self.acoustic_projection = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 32,
                64,
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),
        )


        # ----------------------------------------------------
        # Engineered NVH feature branch
        # ----------------------------------------------------

        self.feature_projection = nn.Sequential(

            nn.Linear(
                engineered_feature_count,
                64,
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                64,
                32,
            ),

            nn.ReLU(),
        )


        # ----------------------------------------------------
        # Fusion
        # ----------------------------------------------------

        self.fusion = nn.Sequential(

            nn.Linear(
                64 + 32,
                64,
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),
        )


        self.rpm_head = nn.Linear(
            64,
            1,
        )


        self.torque_head = nn.Linear(
            64,
            1,
        )


    def forward(
        self,
        logmel,
        engineered_features,
    ):

        acoustic = (
            self.acoustic_encoder(
                logmel
            )
        )

        acoustic = (
            self.acoustic_projection(
                acoustic
            )
        )


        engineered = (
            self.feature_projection(
                engineered_features
            )
        )


        fused = torch.cat(
            [
                acoustic,
                engineered,
            ],
            dim=1,
        )


        shared = self.fusion(
            fused
        )


        rpm = (
            self.rpm_head(
                shared
            )
            .squeeze(-1)
        )


        torque = (
            self.torque_head(
                shared
            )
            .squeeze(-1)
        )


        return rpm, torque
'''


HYBRID_MODEL_MODULE_PATH.write_text(
    textwrap.dedent(
        hybrid_model_code
    ).lstrip(),
    encoding="utf-8",
)


print(
    f"Created:\n"
    f"{HYBRID_MODEL_MODULE_PATH}"
)

Created:
/content/engine-nvh-deep-learning/src/hybrid_multitask_models.py


In [13]:
# Step 98 — Freeze MTL-002 specification
# This is important: we freeze the model before seeing its results.
# Recommendation is to define “material improvement” as at least
# 2% MAE improvement on one target, while not being worse on the other target.

# Step 98.1 — Save the MTL-002 config
# ============================================================
# STEP 98.1 — FREEZE MTL-002 SPECIFICATION
# ============================================================

MTL002_CONFIG_PATH = (
    REPO_DIR
    / "configs"
    / "multitask_hybrid_v001.yaml"
)


mtl002_config = {

    "specification": {

        "model_version":
            "MTL-002",

        "status":
            "frozen_pretraining_spec",
    },

    "experiment_hypothesis":
        (
            "Fuse learned log-mel CNN representations "
            "with the same 41 engineered FEAT-001 NVH "
            "predictors used by the strong conventional "
            "RPM and torque baselines."
        ),

    "population": {

        "source":
            "TORQUE-SPEC-001",

        "train_count":
            7739,

        "validation_count":
            1749,

        "test_count":
            1815,
    },

    "inputs": {

        "acoustic":
            "CNN-001 log-mel representation",

        "engineered":
            "FEAT-001 41 predictors",

        "engineered_feature_count":
            41,

        "engineered_normalization":
            "training-only per-feature mean/std",
    },

    "targets": {

        "rpm":
            "rpm_mean",

        "torque":
            "torque_mean_nm",

        "normalization":
            "MTL-001 training-only target normalization",
    },

    "architecture": {

        "type":
            "hybrid_frequency_aware_multitask_cnn",

        "acoustic_embedding_dim":
            64,

        "engineered_embedding_dim":
            32,

        "fusion_dim":
            64,

        "dropout":
            0.20,

        "heads": [
            "rpm",
            "torque",
        ],
    },

    "training": {

        "rpm_loss":
            "MSELoss_standardized",

        "torque_loss":
            "MSELoss_standardized",

        "rpm_loss_weight":
            0.5,

        "torque_loss_weight":
            0.5,

        "optimizer":
            "AdamW",

        "learning_rate":
            0.001,

        "weight_decay":
            0.0001,

        "batch_size":
            64,

        "maximum_epochs":
            25,

        "early_stopping_patience":
            7,

        "random_seed":
            42,
    },

    "checkpoint_selection": {

        "metric":
            "validation_joint_normalized_mae",
    },

    "test_approval_rule": {

        "must_not_be_worse_than_benchmark_on_either_target":
            True,

        "minimum_relative_mae_improvement_on_at_least_one_target":
            0.02,

        "description":
            (
                "MTL-002 must match or beat both frozen "
                "shared-population conventional benchmark "
                "MAEs and improve at least one target "
                "by >=2%."
            ),
    },

    "test_used_for_tuning":
        False,

    "project_policy":
        (
            "MTL-002 is the final planned model-development "
            "experiment regardless of outcome."
        ),
}


with open(
    MTL002_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:

    yaml.safe_dump(
        mtl002_config,
        file,
        sort_keys=False,
    )


print(
    f"Saved:\n"
    f"{MTL002_CONFIG_PATH}"
)

print(
    "PASS: MTL-002 specification frozen."
)

Saved:
/content/engine-nvh-deep-learning/configs/multitask_hybrid_v001.yaml
PASS: MTL-002 specification frozen.


In [14]:
# Step 99 — Build and smoke-test the full MTL-002 pipeline
# Step 99.1 — Restore HDF5 metadata and shared indices

# ============================================================
# STEP 99.1 — RESTORE CACHE INDICES
# ============================================================

CNN_LOGMEL_DRIVE_PATH = (
    PROJECT_DRIVE
    / "data"
    / "cached_features"
    / "CNN-001"
    / "logmel_v001.h5"
)


with h5py.File(
    CNN_LOGMEL_DRIVE_PATH,
    "r",
) as h5:

    cache_sample_ids = (
        h5["sample_id"]
        .asstr()[:]
    )

    cache_splits = (
        h5["split"]
        .asstr()[:]
    )

    cache_shape = (
        h5["log_mel"].shape
    )


mtl_sample_set = set(
    mtl_population[
        "sample_id"
    ]
)


mtl_mask = np.array(
    [
        sample_id
        in mtl_sample_set

        for sample_id in
        cache_sample_ids
    ]
)


MTL002_TRAIN_INDICES = np.where(
    mtl_mask
    &
    (
        cache_splits == "train"
    )
)[0]


MTL002_VALIDATION_INDICES = np.where(
    mtl_mask
    &
    (
        cache_splits == "validation"
    )
)[0]


MTL002_TEST_INDICES = np.where(
    mtl_mask
    &
    (
        cache_splits == "test"
    )
)[0]


assert len(MTL002_TRAIN_INDICES) == 7739
assert len(MTL002_VALIDATION_INDICES) == 1749
assert len(MTL002_TEST_INDICES) == 1815


LOG_MEL_SHAPE = (
    int(cache_shape[1]),
    int(cache_shape[2]),
)


print(
    "PASS: MTL-002 cache indices restored."
)

PASS: MTL-002 cache indices restored.


In [15]:
# Step 99.2 — Build engineered-feature arrays aligned to HDF5 order

# ============================================================
# STEP 99.2 — BUILD ALIGNED ENGINEERED FEATURE MATRICES
# ============================================================

feature_lookup = (
    mtl002_feature_population
    .set_index(
        "sample_id"
    )
)


def build_feature_matrix(
    cache_indices,
):

    ordered_sample_ids = (
        cache_sample_ids[
            cache_indices
        ]
    )


    matrix = (
        feature_lookup
        .loc[
            ordered_sample_ids,
            MTL002_FEATURE_COLUMNS,
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    matrix = (
        matrix
        -
        feature_mean
        .loc[
            MTL002_FEATURE_COLUMNS
        ]
        .to_numpy(
            dtype=np.float32
        )
    ) / (
        feature_std_safe
        .loc[
            MTL002_FEATURE_COLUMNS
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    return matrix


MTL002_TRAIN_FEATURES = (
    build_feature_matrix(
        MTL002_TRAIN_INDICES
    )
)


MTL002_VALIDATION_FEATURES = (
    build_feature_matrix(
        MTL002_VALIDATION_INDICES
    )
)


assert (
    MTL002_TRAIN_FEATURES.shape
    ==
    (7739, 41)
)

assert (
    MTL002_VALIDATION_FEATURES.shape
    ==
    (1749, 41)
)


assert np.isfinite(
    MTL002_TRAIN_FEATURES
).all()


print(
    "Train feature matrix:",
    MTL002_TRAIN_FEATURES.shape,
)

print(
    "Validation feature matrix:",
    MTL002_VALIDATION_FEATURES.shape,
)

print(
    "PASS: Hybrid feature matrices ready."
)

Train feature matrix: (7739, 41)
Validation feature matrix: (1749, 41)
PASS: Hybrid feature matrices ready.


In [16]:
# Step 99.3 — Restore log-mel normalization

# ============================================================
# STEP 99.3 — RESTORE LOG-MEL NORMALIZATION
# ============================================================

CNN_NORMALIZATION_PATH = (
    REPO_DIR
    / "data"
    / "cnn001_normalization_v001.yaml"
)


with open(
    CNN_NORMALIZATION_PATH,
    "r",
    encoding="utf-8",
) as file:

    cnn_norm = yaml.safe_load(
        file
    )


MTL002_LOGMEL_MEAN = float(
    cnn_norm["log_mel"]["mean"]
)

MTL002_LOGMEL_STD = float(
    cnn_norm["log_mel"]["std"]
)


print(
    "PASS: Log-mel normalization restored."
)

PASS: Log-mel normalization restored.


In [17]:
# Step 99.4 — Mirror HDF5 cache locally

# ============================================================
# STEP 99.4 — MIRROR HDF5 CACHE LOCALLY
# MODERATE I/O TIME
# ============================================================

from shutil import copy2

LOCAL_CACHE_DIR = Path(
    "/content/nvh_runtime_cache/CNN-001"
)

LOCAL_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MTL002_LOCAL_H5_PATH = (
    LOCAL_CACHE_DIR
    / "logmel_v001.h5"
)


copy2(
    CNN_LOGMEL_DRIVE_PATH,
    MTL002_LOCAL_H5_PATH,
)


assert (
    MTL002_LOCAL_H5_PATH.exists()
)

assert (
    MTL002_LOCAL_H5_PATH.stat().st_size
    ==
    CNN_LOGMEL_DRIVE_PATH.stat().st_size
)


CNN_LOGMEL_CACHE_PATH = (
    MTL002_LOCAL_H5_PATH
)


print(
    f"Runtime cache:\n"
    f"{CNN_LOGMEL_CACHE_PATH}"
)

print(
    "PASS: Local HDF5 cache active."
)

Runtime cache:
/content/nvh_runtime_cache/CNN-001/logmel_v001.h5
PASS: Local HDF5 cache active.


In [18]:
# Step 99.5 — Create Dataset objects and DataLoaders

# ============================================================
# STEP 99.5 — CREATE MTL-002 DATASETS / LOADERS
# ============================================================

if str(REPO_DIR) not in sys.path:

    sys.path.insert(
        0,
        str(REPO_DIR),
    )


import src.hybrid_multitask_dataset
import src.hybrid_multitask_models

importlib.reload(
    src.hybrid_multitask_dataset
)

importlib.reload(
    src.hybrid_multitask_models
)


from src.hybrid_multitask_dataset import (
    HybridLogMelFeatureDataset
)

from src.hybrid_multitask_models import (
    HybridRPMTorqueMTL
)

from torch.utils.data import (
    DataLoader
)


mtl002_train_dataset = (
    HybridLogMelFeatureDataset(

        CNN_LOGMEL_CACHE_PATH,

        MTL002_TRAIN_INDICES,

        MTL002_TRAIN_FEATURES,

        MTL002_LOGMEL_MEAN,
        MTL002_LOGMEL_STD,

        MTL_RPM_MEAN,
        MTL_RPM_STD,

        MTL_TORQUE_MEAN,
        MTL_TORQUE_STD,
    )
)


mtl002_validation_dataset = (
    HybridLogMelFeatureDataset(

        CNN_LOGMEL_CACHE_PATH,

        MTL002_VALIDATION_INDICES,

        MTL002_VALIDATION_FEATURES,

        MTL002_LOGMEL_MEAN,
        MTL002_LOGMEL_STD,

        MTL_RPM_MEAN,
        MTL_RPM_STD,

        MTL_TORQUE_MEAN,
        MTL_TORQUE_STD,
    )
)


MTL002_RANDOM_SEED = 42
MTL002_BATCH_SIZE = 64


mtl002_generator = (
    torch.Generator()
)

mtl002_generator.manual_seed(
    MTL002_RANDOM_SEED
)


mtl002_train_loader = DataLoader(

    mtl002_train_dataset,

    batch_size=
        MTL002_BATCH_SIZE,

    shuffle=True,

    generator=
        mtl002_generator,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available(),
)


mtl002_validation_loader = DataLoader(

    mtl002_validation_dataset,

    batch_size=
        MTL002_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available(),
)


print(
    "PASS: MTL-002 DataLoaders created."
)

PASS: MTL-002 DataLoaders created.


In [19]:
# Step 99.6 — Forward-pass smoke test

# ============================================================
# STEP 99.6 — MTL-002 FORWARD-PASS SMOKE TEST
# ============================================================

import random
import torch.nn as nn


random.seed(
    MTL002_RANDOM_SEED
)

np.random.seed(
    MTL002_RANDOM_SEED
)

torch.manual_seed(
    MTL002_RANDOM_SEED
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


X_audio, X_feat, y_rpm, y_torque, _ = (
    next(
        iter(
            mtl002_train_loader
        )
    )
)


smoke_model = (
    HybridRPMTorqueMTL(
        engineered_feature_count=41,
        dropout=0.20,
    )
    .to(
        device
    )
)


with torch.no_grad():

    rpm_out, torque_out = (
        smoke_model(
            X_audio.to(
                device
            ),
            X_feat.to(
                device
            ),
        )
    )


assert rpm_out.shape == y_rpm.shape
assert torque_out.shape == y_torque.shape

assert torch.isfinite(
    rpm_out
).all()

assert torch.isfinite(
    torque_out
).all()


MTL002_PARAMETER_COUNT = sum(
    parameter.numel()
    for parameter
    in smoke_model.parameters()
)


print(
    f"Audio:       {X_audio.shape}"
)

print(
    f"Features:    {X_feat.shape}"
)

print(
    f"RPM output:  {rpm_out.shape}"
)

print(
    f"Torque out:  {torque_out.shape}"
)

print(
    f"Parameters:  {MTL002_PARAMETER_COUNT:,}"
)

print()

print(
    "PASS: MTL-002 complete forward "
    "pipeline validated."
)

print(
    "NO TRAINING HAS OCCURRED."
)

Audio:       torch.Size([64, 1, 64, 59])
Features:    torch.Size([64, 41])
RPM output:  torch.Size([64])
Torque out:  torch.Size([64])
Parameters:  165,762

PASS: MTL-002 complete forward pipeline validated.
NO TRAINING HAS OCCURRED.


In [20]:
# Step 100 — Persist the MTL-002 pre-training state
# Do this before starting training, so you can walk away knowing the
# specification and code are already safely committed.

# Step 100.1 — Create readiness record

# ============================================================
# STEP 100.1 — MTL-002 PRETRAINING READINESS
# ============================================================

MTL002_READINESS_PATH = (
    REPO_DIR
    / "data"
    / "mtl002_pretraining_readiness_v001.yaml"
)


mtl002_readiness = {

    "model_version":
        "MTL-002",

    "specification_frozen":
        True,

    "shared_population_validated":
        True,

    "feature_normalization_frozen":
        True,

    "dataset_module_validated":
        True,

    "model_module_validated":
        True,

    "forward_pass_validated":
        True,

    "parameter_count":
        int(
            MTL002_PARAMETER_COUNT
        ),

    "training_started":
        False,

    "completed_epochs":
        0,

    "test_loader_created":
        False,

    "test_evaluated":
        False,

    "project_policy":
        (
            "Final planned model experiment "
            "regardless of validation outcome."
        ),

    "next_step":
        "Step 101 — MTL-002 training",
}


with open(
    MTL002_READINESS_PATH,
    "w",
    encoding="utf-8",
) as file:

    yaml.safe_dump(
        mtl002_readiness,
        file,
        sort_keys=False,
    )


print(
    f"Saved:\n"
    f"{MTL002_READINESS_PATH}"
)

Saved:
/content/engine-nvh-deep-learning/data/mtl002_pretraining_readiness_v001.yaml
